In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("fraud")
         .getOrCreate())
print(spark.version)

c:\Users\Dell\OneDrive\Desktop\ClusterProject\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.3.0


In [3]:
from pyspark.sql.types import *

schema = StructType([
    StructField("transaction_id", IntegerType()),
    StructField("amount_usd", DoubleType()),
    StructField("merchant_category", StringType()),
    StructField("card_type", StringType()),
    StructField("auth_method", StringType()),
    StructField("channel", StringType()),
    StructField("device_type", StringType()),
    StructField("is_foreign_transaction", BooleanType()),
    StructField("hours_since_last_txn", DoubleType()),
    StructField("txn_count_last_24h", IntegerType()),
    StructField("distance_from_home_km", DoubleType()),
    StructField("card_age_months", IntegerType()),
    StructField("customer_age", IntegerType()),
    StructField("account_balance_usd", DoubleType()),
    StructField("is_new_merchant", BooleanType()),
    StructField("used_vpn", BooleanType()),
    StructField("ip_country_mismatch", BooleanType()),
    StructField("billing_shipping_mismatch", BooleanType()),
    StructField("cvv_retry_count", IntegerType()),
    StructField("velocity_score", DoubleType()),
    StructField("time_of_day_hour", IntegerType()),
    StructField("day_of_week", IntegerType()),
    StructField("is_ai_generated_scam_attempt", BooleanType()),
    StructField("merchant_risk_score", DoubleType()),
    StructField("prior_disputes", IntegerType()),
    StructField("is_fraud", IntegerType()),
])


In [4]:
df = spark.read.option("header", True).schema(schema).csv("../data/credit_card_fraud_2026.csv")
df.printSchema()
print(df.rdd.getNumPartitions())
print(df.count())
df.show(5)

root
 |-- transaction_id: integer (nullable = true)
 |-- amount_usd: double (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- auth_method: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- hours_since_last_txn: double (nullable = true)
 |-- txn_count_last_24h: integer (nullable = true)
 |-- distance_from_home_km: double (nullable = true)
 |-- card_age_months: integer (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- account_balance_usd: double (nullable = true)
 |-- is_new_merchant: boolean (nullable = true)
 |-- used_vpn: boolean (nullable = true)
 |-- ip_country_mismatch: boolean (nullable = true)
 |-- billing_shipping_mismatch: boolean (nullable = true)
 |-- cvv_retry_count: integer (nullable = true)
 |-- velocity_score: double (nullable = true)
 |-- time_of_day_hour: integer (nullable = 

In [5]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)


-RECORD 0---------------------------
 transaction_id               | 0   
 amount_usd                   | 0   
 merchant_category            | 0   
 card_type                    | 0   
 auth_method                  | 0   
 channel                      | 0   
 device_type                  | 0   
 is_foreign_transaction       | 0   
 hours_since_last_txn         | 0   
 txn_count_last_24h           | 0   
 distance_from_home_km        | 0   
 card_age_months              | 0   
 customer_age                 | 0   
 account_balance_usd          | 0   
 is_new_merchant              | 0   
 used_vpn                     | 0   
 ip_country_mismatch          | 0   
 billing_shipping_mismatch    | 0   
 cvv_retry_count              | 0   
 velocity_score               | 0   
 time_of_day_hour             | 0   
 day_of_week                  | 0   
 is_ai_generated_scam_attempt | 0   
 merchant_risk_score          | 0   
 prior_disputes               | 0   
 is_fraud                     | 0   



In [6]:

df.groupBy("is_fraud").count().show()
print(df.repartition(4).rdd.getNumPartitions())

+--------+-----+
|is_fraud|count|
+--------+-----+
|       1|  339|
|       0|19661|
+--------+-----+

4


In [7]:
df.describe("amount_usd", "customer_age", "account_balance_usd", "velocity_score").show()
df.select("merchant_category").distinct().show()
df.filter(F.col("customer_age") < 18).count()

+-------+------------------+-----------------+-------------------+------------------+
|summary|        amount_usd|     customer_age|account_balance_usd|    velocity_score|
+-------+------------------+-----------------+-------------------+------------------+
|  count|             20000|            20000|              20000|             20000|
|   mean|132.42459699999958|          49.6694| 3316.6625980000017|19.807735000000058|
| stddev| 256.9636655626206|18.49403160752827|  4350.720948381676|12.366300899609111|
|    min|               1.0|               18|              52.05|               0.0|
|    max|           6872.69|               81|          127125.86|              74.4|
+-------+------------------+-----------------+-------------------+------------------+

+-----------------+
|merchant_category|
+-----------------+
|           Gaming|
|    Online Retail|
|       Healthcare|
|        Groceries|
|           Travel|
|      Electronics|
|  Crypto Exchange|
|        Utilities|
|    

0

In [8]:
df.createOrReplaceTempView("txns")

spark.sql("""
    SELECT channel, COUNT(*) AS n, ROUND(AVG(amount_usd), 2) AS avg_amount
    FROM txns
    GROUP BY channel
    ORDER BY n DESC
""").show()

+-----------+----+----------+
|    channel|   n|avg_amount|
+-----------+----+----------+
|     Online|6810|    134.92|
|        POS|5191|    128.32|
|Contactless|3400|    131.31|
|     In-App|3225|    137.96|
|        ATM|1374|    125.36|
+-----------+----+----------+



In [9]:
def fraud_rate_by(col):
    return (df.groupBy(col)
              .agg(F.count("*").alias("n"),
                   F.sum("is_fraud").alias("frauds"),
                   F.round(F.avg("is_fraud") * 100, 2).alias("fraud_pct"))
              .orderBy(F.desc("fraud_pct")))

for c in ["merchant_category", "channel", "auth_method", "cvv_retry_count",
          "ip_country_mismatch", "is_ai_generated_scam_attempt", "is_foreign_transaction"]:
    print(c)
    fraud_rate_by(c).show()

merchant_category
+-----------------+----+------+---------+
|merchant_category|   n|frauds|fraud_pct|
+-----------------+----+------+---------+
|  Crypto Exchange| 561|    28|     4.99|
|       Gift Cards| 617|    27|     4.38|
|           Gaming|1036|    38|     3.67|
|      Electronics|2032|    48|     2.36|
|        Streaming|1010|    21|     2.08|
|        Utilities|1330|    25|     1.88|
|       Healthcare|1237|    16|     1.29|
|        Groceries|3242|    40|     1.23|
|             Fuel|1565|    19|     1.21|
|      Restaurants|2534|    29|     1.14|
|           Travel|1420|    16|     1.13|
|    Online Retail|3416|    32|     0.94|
+-----------------+----+------+---------+

channel
+-----------+----+------+---------+
|    channel|   n|frauds|fraud_pct|
+-----------+----+------+---------+
|     Online|6810|   135|     1.98|
|        ATM|1374|    25|     1.82|
|     In-App|3225|    52|     1.61|
|        POS|5191|    79|     1.52|
|Contactless|3400|    48|     1.41|
+-----------+

In [10]:
(df.groupBy("is_fraud")
   .agg(F.count("*").alias("n"),
        F.round(F.avg("amount_usd"), 1).alias("avg_amount"),
        F.round(F.avg("velocity_score"), 1).alias("avg_velocity"),
        F.round(F.avg("merchant_risk_score"), 1).alias("avg_merchant_risk"),
        F.round(F.avg("distance_from_home_km"), 1).alias("avg_distance"),
        F.round(F.avg("hours_since_last_txn"), 1).alias("avg_hours_since_last"),
        F.round(F.avg("txn_count_last_24h"), 1).alias("avg_txn_24h"))
   .show())

+--------+-----+----------+------------+-----------------+------------+--------------------+-----------+
|is_fraud|    n|avg_amount|avg_velocity|avg_merchant_risk|avg_distance|avg_hours_since_last|avg_txn_24h|
+--------+-----+----------+------------+-----------------+------------+--------------------+-----------+
|       1|  339|     181.2|        29.0|             51.4|        21.4|                 7.0|        4.0|
|       0|19661|     131.6|        19.6|             37.2|        22.2|                 9.0|        3.2|
+--------+-----+----------+------------+-----------------+------------+--------------------+-----------+



In [11]:
fraud_rate_by("time_of_day_hour").orderBy("time_of_day_hour").show(24)

+----------------+---+------+---------+
|time_of_day_hour|  n|frauds|fraud_pct|
+----------------+---+------+---------+
|               0|808|    13|     1.61|
|               1|887|    20|     2.25|
|               2|814|    28|     3.44|
|               3|815|    23|     2.82|
|               4|826|    20|     2.42|
|               5|821|    33|     4.02|
|               6|840|     8|     0.95|
|               7|851|     9|     1.06|
|               8|785|     9|     1.15|
|               9|820|    10|     1.22|
|              10|824|    17|     2.06|
|              11|810|    18|     2.22|
|              12|840|     7|     0.83|
|              13|904|     7|     0.77|
|              14|815|    11|     1.35|
|              15|876|    11|     1.26|
|              16|789|    14|     1.77|
|              17|845|    15|     1.78|
|              18|794|     7|     0.88|
|              19|872|    12|     1.38|
|              20|865|    17|     1.97|
|              21|821|     9|      1.1|


In [14]:
def fraud_rate_by(col):
    return (df.groupBy(col)
              .agg(F.count("*").alias("n"),
                   F.sum("is_fraud").alias("frauds"),
                   F.round(F.avg("is_fraud") * 100, 2).alias("fraud_pct"))
              .orderBy(F.desc("fraud_pct")))

In [15]:
fraud_rate_by("cvv_retry_count").show()
fraud_rate_by("ip_country_mismatch").show()
fraud_rate_by("auth_method").show()


+---------------+-----+------+---------+
|cvv_retry_count|    n|frauds|fraud_pct|
+---------------+-----+------+---------+
|              2|  254|    41|    16.14|
|              3|   15|     2|    13.33|
|              1| 3073|   143|     4.65|
|              0|16658|   153|     0.92|
+---------------+-----+------+---------+

+-------------------+-----+------+---------+
|ip_country_mismatch|    n|frauds|fraud_pct|
+-------------------+-----+------+---------+
|               true| 1166|    91|      7.8|
|              false|18834|   248|     1.32|
+-------------------+-----+------+---------+

+-----------------+----+------+---------+
|      auth_method|   n|frauds|fraud_pct|
+-----------------+----+------+---------+
|No Authentication|2388|   106|     4.44|
|              OTP|4792|    75|     1.57|
|        3D Secure|5554|    86|     1.55|
|              PIN|4173|    48|     1.15|
|        Biometric|3093|    24|     0.78|
+-----------------+----+------+---------+



In [16]:
(df.groupBy("is_fraud")
   .agg(F.count("*").alias("n"),
        F.round(F.avg("amount_usd"), 1).alias("avg_amount"),
        F.round(F.avg("velocity_score"), 1).alias("avg_velocity"),
        F.round(F.avg("merchant_risk_score"), 1).alias("avg_merchant_risk"),
        F.round(F.avg("distance_from_home_km"), 1).alias("avg_distance"),
        F.round(F.avg("hours_since_last_txn"), 1).alias("avg_hours_since_last"),
        F.round(F.avg("txn_count_last_24h"), 1).alias("avg_txn_24h"))
   .show())

+--------+-----+----------+------------+-----------------+------------+--------------------+-----------+
|is_fraud|    n|avg_amount|avg_velocity|avg_merchant_risk|avg_distance|avg_hours_since_last|avg_txn_24h|
+--------+-----+----------+------------+-----------------+------------+--------------------+-----------+
|       1|  339|     181.2|        29.0|             51.4|        21.4|                 7.0|        4.0|
|       0|19661|     131.6|        19.6|             37.2|        22.2|                 9.0|        3.2|
+--------+-----+----------+------------+-----------------+------------+--------------------+-----------+



In [17]:
spark.sql("""
    SELECT merchant_category,
           COUNT(*) AS n,
           ROUND(AVG(is_fraud) * 100, 2) AS fraud_pct
    FROM txns
    GROUP BY merchant_category
    HAVING COUNT(*) >= 100
    ORDER BY fraud_pct DESC
""").show()

+-----------------+----+---------+
|merchant_category|   n|fraud_pct|
+-----------------+----+---------+
|  Crypto Exchange| 561|     4.99|
|       Gift Cards| 617|     4.38|
|           Gaming|1036|     3.67|
|      Electronics|2032|     2.36|
|        Streaming|1010|     2.08|
|        Utilities|1330|     1.88|
|       Healthcare|1237|     1.29|
|        Groceries|3242|     1.23|
|             Fuel|1565|     1.21|
|      Restaurants|2534|     1.14|
|           Travel|1420|     1.13|
|    Online Retail|3416|     0.94|
+-----------------+----+---------+



# Credit Card Fraud Analysis with PySpark

**Dataset:** 20,000 card transactions, 26 columns, of which 339 are fraudulent (1.7%). A null check found no missing values, and the schema was defined explicitly when loading.

## Findings

1. **Strongest single signal: `cvv_retry_count`.** Fraud rate is 0.92% with 0 CVV retries (n = 16,658), 4.65% with 1 retry (n = 3,073) and 16.14% with 2 retries (n = 254), about 17 times higher than with none. Transactions with at least one retry are 3,342 of 20,000 (about 17%) but contain 186 of the 339 frauds (about 55%). The 3-retry group (13.33%) has only 15 transactions, so it is too small to rely on.
   Other strong signals: `ip_country_mismatch` = true has a fraud rate of 7.80% (n = 1,166) versus 1.32% (n = 18,834), and `auth_method` = No Authentication has 4.44% (n = 2,388) versus 0.78% for Biometric (n = 3,093).

2. **Riskiest merchant category:** Crypto Exchange at 4.99% (n = 561), versus 0.94% for Online Retail (n = 3,416). Gift Cards (4.38%) and Gaming (3.67%) are next. The lower categories are all between about 0.9% and 2.4%.

3. **Combined flags:** counting six red flags per transaction (IP country mismatch, billing/shipping mismatch, VPN use, foreign transaction, new merchant, AI-generated scam attempt), fraud rises at every step: 0.46% with 0 flags (n = 11,871), 1.98% with 1 (n = 6,480), 6.88% with 2 (n = 1,425) and 24.53% with 3 (n = 212). The 12 transactions with 4 flags were 50% fraud, but that sample is too small to rely on.

4. **Time of day:** hours 0 to 5 have a higher fraud rate (2.76% combined, from 137 frauds in 4,971 transactions) than hours 6 to 23 (1.34%, from 202 frauds in 15,029). That is about twice the rate, and the six early hours hold roughly 40% of all fraud. Individual hours are noisy (about 800 transactions each), so the pattern is the block of early hours, not any single hour.

5. **Numeric features:** fraudulent transactions have a higher average amount ($181.2 vs $131.6), velocity score (29.0 vs 19.6), merchant risk score (51.4 vs 37.2) and transactions in the last 24 hours (4.0 vs 3.2), and a shorter time since the last transaction (7.0 vs 9.0 hours). Average distance from home is almost the same for both groups (21.4 vs 22.2 km), so it is probably a weak feature. These are averages, and `amount_usd` is skewed by a few very large transactions.

## Takeaways

- No single signal is enough, but fraud risk climbs sharply as red flags accumulate.
- Only 1.7% of transactions are fraud, so accuracy would be a misleading metric: predicting "not fraud" every time would be right about 98.3% of the time. The model will be judged with PR-AUC and recall instead.
- Next step: use these signals as features in a Spark ML pipeline.
-Finding 1 picks cvv_retry_count over ip_country_mismatch: it has the larger gap between groups (17× vs 6×) and a decent sample size at 2 retries (n = 254)

## Limitations

- The dataset has no date or customer ID columns, so trends over time and per-customer history cannot be analyzed.
- 20,000 rows fit easily in pandas. Spark was used here to practice a scalable workflow that works unchanged on much larger data.